# Football Player Positioning AI - LSTM Training

This notebook trains per-player LSTM models on Google Colab with free GPU.

**Steps:**
1. Clone repo & install dependencies
2. Download Metrica Sports data
3. Run preprocessing pipeline
4. Run feature engineering
5. Train LSTM model (single player)
6. Verify & visualize results

## 0. Check GPU

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU')

## 1. Clone Repository

In [ ]:
import os
if os.path.exists('football-positioning-ai'):
    print('[DEBUG] Repo already cloned, pulling latest...')
    !cd football-positioning-ai && git pull
else:
    print('[DEBUG] Cloning repo...')
    !git clone https://github.com/Sysus611/football-positioning-ai.git
%cd football-positioning-ai
print(f'[DEBUG] Working dir: {os.getcwd()}')
print(f'[DEBUG] Files: {os.listdir(".")}')
!pip install pandas numpy pyarrow scipy pyyaml matplotlib -q
print('[OK] Dependencies installed')

## 2. Download Metrica Sports Data

In [ ]:
import os
import urllib.request

BASE_URL = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data"

# Local filename -> GitHub actual filename mapping
FILES = {
    'game1': {
        'tracking_home.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv',
    },
    'game2': {
        'tracking_home.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Home_Team.csv',
        'tracking_away.csv': 'Sample_Game_2/Sample_Game_2_RawTrackingData_Away_Team.csv',
    },
    'game3': {
        'tracking.txt': 'Sample_Game_3/Sample_Game_3_tracking.txt',
        'metadata.xml': 'Sample_Game_3/Sample_Game_3_metadata.xml',
        'metadata.json': 'Sample_Game_3/Sample_Game_3_events.json',
    },
}

print('[DEBUG] Starting data download...')
for game, file_map in FILES.items():
    game_dir = f'data/raw/metrica/{game}'
    os.makedirs(game_dir, exist_ok=True)
    for local_name, remote_path in file_map.items():
        url = f'{BASE_URL}/{remote_path}'
        dest = f'{game_dir}/{local_name}'
        if os.path.exists(dest):
            size_mb = os.path.getsize(dest) / (1024*1024)
            print(f'  [SKIP] {game}/{local_name} already exists ({size_mb:.1f} MB)')
            continue
        print(f'  [DOWNLOAD] {remote_path}...', end='')
        urllib.request.urlretrieve(url, dest)
        size_mb = os.path.getsize(dest) / (1024*1024)
        print(f' OK ({size_mb:.1f} MB)')

# Verify
print('\n[DEBUG] Downloaded files:')
for game in FILES:
    game_dir = f'data/raw/metrica/{game}'
    files_in_dir = os.listdir(game_dir)
    total_mb = sum(os.path.getsize(os.path.join(game_dir, f)) for f in files_in_dir) / (1024*1024)
    print(f'  {game}: {files_in_dir} ({total_mb:.1f} MB)')
print('[OK] Download complete!')

## 3. Run Preprocessing

In [ ]:
import time
print('[DEBUG] Starting preprocessing pipeline...')
t0 = time.time()
!python src/preprocessing/preprocess.py
elapsed = time.time() - t0
print(f'\n[OK] Preprocessing done in {elapsed:.0f}s')

# Verify outputs
import os
pdir = 'data/processed'
if os.path.exists(pdir):
    for f in sorted(os.listdir(pdir)):
        size = os.path.getsize(os.path.join(pdir, f)) / (1024*1024)
        print(f'  [OUTPUT] {f}: {size:.1f} MB')
else:
    print('[ERROR] No processed output found!')

## 4. Feature Engineering

In [ ]:
import time
print('[DEBUG] Starting feature engineering...')
t0 = time.time()
!python src/features/build_features.py
elapsed = time.time() - t0
print(f'\n[OK] Feature engineering done in {elapsed:.0f}s')

# Verify outputs
import os
tdir = 'data/tensors'
if os.path.exists(tdir):
    files = sorted(os.listdir(tdir))
    total_mb = sum(os.path.getsize(os.path.join(tdir, f)) for f in files) / (1024*1024)
    print(f'  [OUTPUT] {len(files)} player datasets, total {total_mb:.0f} MB')
    for f in files:
        size = os.path.getsize(os.path.join(tdir, f)) / (1024*1024)
        print(f'    {f}: {size:.1f} MB')
else:
    print('[ERROR] No tensor output found!')

## 5. Train Single Player

Only train `home_11` first for quick verification. Change the player ID below to train a different player.

After verification, run `!python src/training/train.py` (without player ID) to train all players.

In [ ]:
import time

# ===== Change this to train a different player =====
PLAYER_ID = 'home_11'
# ===================================================

print(f'[DEBUG] Training single player: {PLAYER_ID}')
print(f'[DEBUG] Using device: {"cuda" if torch.cuda.is_available() else "cpu"}')
t0 = time.time()
!python src/training/train.py {PLAYER_ID}
elapsed = time.time() - t0
print(f'\n[OK] Training done in {elapsed:.0f}s ({elapsed/60:.1f} min)')

## 6. Check Results

In [ ]:
import os
import torch

model_dir = 'data/models'
if os.path.exists(model_dir):
    models = sorted(os.listdir(model_dir))
    print(f'[RESULT] Trained {len(models)} models:\n')
    print(f'  {"Player":<20s} | {"Val Loss":<12s} | {"~Dist(m)":<10s} | {"Epoch":<6s}')
    print(f'  {"-"*20}-+-{"-"*12}-+-{"-"*10}-+-{"-"*6}')
    for m in models:
        path = os.path.join(model_dir, m)
        ckpt = torch.load(path, map_location='cpu', weights_only=False)
        val_loss = ckpt['best_val_loss']
        dist_m = (val_loss ** 0.5) * ((105 + 68) / 2)
        print(f"  {m:20s} | {val_loss:10.6f}   | {dist_m:7.2f}m   | {ckpt['epoch']}")
else:
    print('[ERROR] No models found!')

## 7. Quick Visualization: Predict vs Actual

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')
from src.model.lstm_baseline import PlayerLSTM

# Load model
print(f'[DEBUG] Loading model for {PLAYER_ID}...')
ckpt = torch.load(f'data/models/{PLAYER_ID}.pt', map_location='cpu', weights_only=False)
model = PlayerLSTM(input_dim=ckpt['input_dim'], pred_frames=ckpt['pred_frames'])
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'[DEBUG] Model loaded: val_loss={ckpt["best_val_loss"]:.6f}, epoch={ckpt["epoch"]}')

# Load test data
data = np.load(f'data/tensors/{PLAYER_ID}.npz')
X, Y = data['X'], data['Y']
print(f'[DEBUG] Data loaded: X={X.shape}, Y={Y.shape}')

# Pick 4 random samples
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f'{PLAYER_ID} - Predicted vs Actual Trajectory', fontsize=14)

for ax_idx, ax in enumerate(axes.flat):
    idx = np.random.randint(len(X))
    x_input = torch.from_numpy(X[idx:idx+1])

    with torch.no_grad():
        pred = model(x_input).numpy()[0]

    actual = Y[idx]
    obs_x = X[idx, :, -4]
    obs_y = X[idx, :, -3]

    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(1.05, -0.05)
    ax.set_facecolor('#2d5a27')
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor='white', linewidth=1))
    ax.axvline(x=0.5, color='white', linewidth=0.5, alpha=0.5)

    ax.plot(obs_x, obs_y, 'c-', linewidth=1.5, alpha=0.6, label='Observation (3s)')
    ax.plot(actual[:, 0], actual[:, 1], 'w-', linewidth=2, label='Actual')
    ax.plot(pred[:, 0], pred[:, 1], 'r--', linewidth=2, label='Predicted')

    ax.scatter(obs_x[0], obs_y[0], c='cyan', s=50, zorder=5, marker='s')
    ax.scatter(actual[-1, 0], actual[-1, 1], c='white', s=80, zorder=5, marker='*')
    ax.scatter(pred[-1, 0], pred[-1, 1], c='red', s=80, zorder=5, marker='*')

    err = np.sqrt(((pred - actual)**2).sum(axis=1)).mean() * ((105+68)/2)
    ax.set_title(f'Sample #{idx} (avg err: {err:.1f}m)', color='white')
    ax.legend(fontsize=8, loc='upper right')
    ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('prediction_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Saved: prediction_visualization.png')

## 8. Download Model

In [ ]:
from google.colab import files
files.download(f'data/models/{PLAYER_ID}.pt')
print(f'[OK] Downloading {PLAYER_ID}.pt')